In [1]:
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-core==1.2.6
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0
!pip install -q rank-bm25
!pip install --upgrade torchao>=0.16.0
!pip install -U ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.5/580.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
import os
import torch
import re
from typing import List, Dict, Any
from ddgs import DDGS
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_classic.chains import HypotheticalDocumentEmbedder
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from IPython.display import display, Markdown, HTML
import time
import warnings
warnings.simplefilter("ignore", category=DeprecationWarning)
warnings.simplefilter("ignore", category=RuntimeWarning)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [3]:
CHUNK_SIZE_PARENT = 800
CHUNK_SIZE_CHILD = 200
CROSS_ENCODER_MODEL = "BAAI/bge-reranker-base"
EMBEDDING_MODEL = "BAAI/bge-m3"
MODEL_NAME = "hendrimardani/fine-tuned-model-indonesian-GRPO"

In [4]:
!gdown --folder "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql"

Retrieving folder contents
Processing file 1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU PP Nomor 5 Tahun 2021.pdf
Processing file 1trvqHE72Anu8MhsNvvYJHieWqbao2BCy PP Nomor 35 Tahun 2021.pdf
Processing file 1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT PP Nomor 51 Tahun 2023.pdf
Processing file 1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s UU Nomor 6 Tahun 2023.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU
To: /kaggle/working/Document RAG/PP Nomor 5 Tahun 2021.pdf
100%|██████████████████████████████████████| 17.1M/17.1M [00:00<00:00, 20.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1trvqHE72Anu8MhsNvvYJHieWqbao2BCy
To: /kaggle/working/Document RAG/PP Nomor 35 Tahun 2021.pdf
100%|██████████████████████████████████████| 2.52M/2.52M [00:00<00:00, 10.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT
To: /kaggle/working/Doc

In [5]:
# Konfigurasi Kuantisasi 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME)


config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=False,             
    max_new_tokens=1500,
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [7]:
class MetadataEnricher:
    def __init__(self, doc_title: str):
        self.doc_title = doc_title

    def _is_true_article_header(self, text_before: str, text_after: str) -> bool:
        """
        Mengecek apakah 'Pasal X' benar-benar kepala pasal atau hanya penyebutan dalam teks.
        """
        # 1. Cek teks SETELAH "Pasal X" pada baris yang sama (untuk kasus inline seperti "Pasal 27 ayat (2)")
        first_line_after = text_after.strip().split('\n')[0].lower()
        inline_body_keywords = ['ayat', 'huruf', 'angka']
        for kw in inline_body_keywords:
            if first_line_after.startswith(kw) or f" {kw} " in f" {first_line_after} ":
                return False

        # 2. Cek HANYA KATA TEPAT SEBELUM "Pasal X" (untuk rujukan seperti "dalam Pasal...", "pada Pasal...")
        words = text_before.strip().split()
        if words:
            last_word = words[-1].lower()
            clean_last_word = re.sub(r'[^\w]', '', last_word)
            
            # Daftar kata preposisi/penghubung yang TEPAT BERADA di depan kata Pasal jika itu rujukan
            reference_prepositions = {'dalam', 'pada', 'menurut', 'sebagaimana', 'ketentuan', 'pasal', 'ayat'}
            if clean_last_word in reference_prepositions:
                return False         
        return True

    def process_and_enrich(self, chunks: List[str]) -> List[Document]:
        # Ekstrak informasi dari judul dokumen (Contoh: "Undang-Undang Nomor 11 Tahun 2020")
        doc_title_string_match = re.search(r'(\w+\s[\w\d\s]*)(Nomor)\s(\d+)\s(\w+)\s(\d{4})', self.doc_title, re.IGNORECASE)
        
        legal_level = "Dokumen Hukum"
        current_number = "-"
        current_year = "2024"
        if doc_title_string_match:
            legal_level, _, current_number, _, current_year = doc_title_string_match.groups()

        documents = []
        current_bab = "Bab Umum"
        current_bagian = "Bagian Umum"
        current_paragraf = "-"
        current_pasal = "Pasal Umum"

        # Iterasi setiap elemen list (di mana indeks merepresentasikan halaman dokumen)
        for page_idx, page_text in enumerate(chunks):
            current_page = page_idx + 1  # Nomor halaman (1, 2, 3, dst.)
            
            # 1. Level 1: Pisahkan berdasarkan BAB di halaman ini
            bab_sections = re.split(r'(?=BAB\s+[IVXLCDM]+)', page_text, flags=re.IGNORECASE)
            
            for bab_sec in bab_sections:
                if not bab_sec.strip():
                    continue
                    
                # Update nama BAB jika ditemukan di bagian ini
                bab_match = re.search(r'(BAB\s+[IVXLCDM]+[^\n]*)', bab_sec, flags=re.IGNORECASE)
                if bab_match:
                    current_bab = bab_match.group(1).strip()
                
                # 2. Level 2: Pisahkan berdasarkan Bagian (Opsional)
                bagian_sections = re.split(r'(?=Bagian\s+[A-Za-z]+)', bab_sec, flags=re.IGNORECASE)
                
                for bag_sec in bagian_sections:
                    if not bag_sec.strip():
                        continue
                        
                    # Update nama Bagian jika ditemukan
                    bag_match = re.search(r'(Bagian\s+ke[A-Za-z]+[^\n]*)', bag_sec, flags=re.IGNORECASE)
                    if bag_match:
                        current_bagian = bag_match.group(1).strip()
                    
                    # Reset paragraf ke "-" setiap kali masuk Bagian baru (karena opsional)
                    current_paragraf = "-"
                    
                    # 3. Level 3: Pisahkan berdasarkan Paragraf (Opsional)
                    paragraf_sections = re.split(r'(?=Paragraf\s+\d+|Paragraf\s+[IVXLCDM]+)', bag_sec, flags=re.IGNORECASE)
                    
                    for para_sec in paragraf_sections:
                        if not para_sec.strip():
                            continue
                            
                        # Update nama Paragraf jika ditemukan
                        para_match = re.search(r'(Paragraf\s+\d+|Paragraf\s+[IVXLCDM]+[^\n]*)', para_sec, flags=re.IGNORECASE)
                        if para_match:
                            current_paragraf = para_match.group(1).strip()
                        
                        # 4. Level 4: Pisahkan berdasarkan Pasal dengan Smart Filter
                        pattern = re.compile(r'(?:^|\n)\s*(Pasal\s+\d+)', re.IGNORECASE)
                        matches = list(pattern.finditer(para_sec))
                        
                        if not matches:
                            continue
                            
                        valid_splits = []
                        for i, m in enumerate(matches):
                            start_pos = m.start()
                            end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(para_sec)
                            
                            text_before = para_sec[:start_pos]
                            text_after = para_sec[m.end():end_pos]
                            
                            # Validasi apakah ini benar-benar kepala pasal atau hanya referensi kalimat
                            if self._is_true_article_header(text_before, text_after):
                                valid_splits.append(m)
                                
                        if not valid_splits:
                            valid_splits = [matches[0]] # Fallback jika semua terdeteksi sebagai referensi
                            
                        # Ekstrak tiap blok pasal yang valid
                        for idx, m in enumerate(valid_splits):
                            start_pos = m.start()
                            end_pos = valid_splits[idx + 1].start() if idx + 1 < len(valid_splits) else len(para_sec)
                            
                            p_sec = para_sec[start_pos:end_pos]
                            
                            pasal_match = re.search(r'(Pasal\s+\d+[^:\n]*)', p_sec, flags=re.IGNORECASE)
                            if pasal_match:
                                current_pasal = pasal_match.group(1).strip()
                            
                            content = p_sec.strip()
                            if content:
                                context_prefix = f"[{self.doc_title} | {current_bab} | {current_bagian} | {current_paragraf} | {current_pasal}]"
                                enriched_text = f"{context_prefix}\n{content}"
                                documents.append(
                                    Document(
                                        metadata={
                                            "halaman": current_page,
                                            "dokumen": self.doc_title,
                                            "tingkat_hukum": legal_level,
                                            "nomor": current_number,
                                            "tahun": current_year,
                                            "bab": current_bab,
                                            "bagian": current_bagian,
                                            "paragraf": current_paragraf,
                                            "pasal": current_pasal
                                        },
                                        page_content=enriched_text,
                                    )
                                )
                                
        return documents

In [8]:
def enrichment_doc_process(file_paths):
    enrichment_documents = []
    total_pages = 0
    
    for file_path in file_paths:
        loader = PyMuPDFLoader(file_path)
        raw_documents = loader.load()
        total_pages += len(raw_documents)
        
        print(f"Berhasil memuat {len(raw_documents)} halaman dari PDF.")
        raw_chunks = [doc.page_content for doc in raw_documents]
        
        metadata_enricher = MetadataEnricher(file_path)
        enriched_chunks = metadata_enricher.process_and_enrich(raw_chunks)
        
        for item in enriched_chunks:
            doc = Document(
                metadata=item.metadata,
                page_content=item.page_content
            )
            enrichment_documents.append(doc)
    
    print(f"Total halaman {total_pages} halaman")
    return enrichment_documents

In [9]:
file_paths = [
    "/kaggle/working/Document RAG/PP Nomor 35 Tahun 2021.pdf",
    "/kaggle/working/Document RAG/PP Nomor 5 Tahun 2021.pdf",
    "/kaggle/working/Document RAG/PP Nomor 51 Tahun 2023.pdf",
    "/kaggle/working/Document RAG/UU Nomor 6 Tahun 2023.pdf"
]


In [10]:
enrichment_documents = enrichment_doc_process(file_paths)
print()

for document in enrichment_documents[:1]:
    print(document.metadata)
    print(document.page_content)
    print()

Berhasil memuat 56 halaman dari PDF.
Berhasil memuat 739 halaman dari PDF.
Berhasil memuat 27 halaman dari PDF.
Berhasil memuat 1127 halaman dari PDF.
Total halaman 1949 halaman

{'halaman': 1, 'dokumen': '/kaggle/working/Document RAG/PP Nomor 35 Tahun 2021.pdf', 'tingkat_hukum': 'PP ', 'nomor': '35', 'tahun': '2021', 'bab': 'Bab Umum', 'bagian': 'Bagian Umum', 'paragraf': '-', 'pasal': 'Pasal 185 huruf b Undang-Undang Nomor 11 Tahun 2O2O'}
[/kaggle/working/Document RAG/PP Nomor 35 Tahun 2021.pdf | Bab Umum | Bagian Umum | - | Pasal 185 huruf b Undang-Undang Nomor 11 Tahun 2O2O]
Pasal 185 huruf b Undang-Undang Nomor 11 Tahun 2O2O
tentang Cipta Kerja, perlu menetapkan peraturan
Pemerintah tentang Perjanjian Kerja Waktu Tertentu, Alih
Daya, Waktu Kerja dan Waktu Istirahat, dan pemutusan
Hubungan Kerja;
1. Pasal 5 ayat (21 Undang-Undang Dasar Negara
Republik Indonesia Tahun l94S;
2. Undang-Undang Nomor 13 Tahun 2OO3 tentang
Ketenagakerjaan (Lembaran Negara Republik
Indonesia Tahun 2003 No

In [11]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE_PARENT)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE_CHILD)
base_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

... (2 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [12]:
hyde_text_generation_pipeline = pipeline(
    model=model,                   
    tokenizer=tokenizer,           
    task="text-generation",
    temperature=0.7,
    num_return_sequences=2,          # Jawaban halusinasi 2 kalimat
    do_sample=True,
    repetition_penalty=1.15,
    max_new_tokens=128,              # Cukup 128 untuk HyDE
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
)
hyde_llm = HuggingFacePipeline(pipeline=hyde_text_generation_pipeline)

hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    hyde_llm,                       
    base_embeddings,
    prompt_key="web_search"
)

Device set to use cuda:0


In [13]:
vectorstore = Chroma.from_documents(
    documents=enrichment_documents,
    embedding=hyde_embeddings,
    collection_name="split_parents",
    persist_directory="./chroma_db",
)

In [14]:
# Store untuk Parent
docstore = InMemoryByteStore()

In [15]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="similarity",
    search_kwargs={"k": 3}
)

batch_size = 200
for i in range(0, len(enrichment_documents), batch_size):
    print(f"batch ke : {i}")
    batched = enrichment_documents[i: i + batch_size]
    parent_retriever.add_documents(batched)
    
print("Data berhasil disimpan kedalam vector store.")

batch ke : 0
batch ke : 200
batch ke : 400
batch ke : 600
batch ke : 800
batch ke : 1000
batch ke : 1200
batch ke : 1400
batch ke : 1600
batch ke : 1800
batch ke : 2000
batch ke : 2200
batch ke : 2400
batch ke : 2600
batch ke : 2800
batch ke : 3000
Data berhasil disimpan kedalam vector store.


In [16]:
bm25_retriever = BM25Retriever.from_documents(enrichment_documents)
bm25_retriever.k = 3

In [17]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=[0.5, 0.5] # 50% fokus ke kata kunci, 50% fokus ke makna kalimat
)

In [18]:
def normalize_to_roman(text: str) -> str:
    """Mengubah penyebutan 'Bab [1-20]' menjadi bentuk angka romawi."""
    arabic_to_roman_map = {
        "1": "I", "2": "II", "3": "III", "4": "IV", "5": "V",
        "6": "VI","7": "VII","8": "VIII","9": "IX","10": "X",
        "11": "XI", "12": "XII", "13": "XIII", "14": "XIV",
        "15": "XV", "16": "XVI", "17": "XVII","18": "XVIII",
        "19": "XIX", "20": "XX",
    }
    text_lower = text.lower()
    pattern = r"bab\s+(\d{1,2})\b"

    def replace_match(match):
        num_str = match.group(1)
        if num_str in arabic_to_roman_map:
            prefix = match.group(0).split()[0]
            return f"{prefix} {arabic_to_roman_map[num_str]}"
        return match.group(0)  # Kembalikan aslinya jika di luar rentang 1-20

    return re.sub(pattern, replace_match, text_lower)

In [19]:
cross_encoder_model = HuggingFaceCrossEncoder(model_name=CROSS_ENCODER_MODEL)
cross_encoder_reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=3)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=cross_encoder_reranker,
    base_retriever=hybrid_retriever,
)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [20]:
def generate_dynamic_filter(query: str) -> dict:
    """
    Mengekstrak entitas hukum dan memformatnya menggunakan operator ChromaDB ($eq / $and)
    tanpa menggunakan operator $regex yang tidak didukung.
    """
    query_lower = query.lower()
    filter_conditions = []
    
    # 1. Deteksi BAB (Contoh: "bab 3" atau "bab iii")
    bab_match = re.search(r'\bbab\s+([ivxlcdm]+|\d+)\b', query_lower)
    if bab_match:
        val = bab_match.group(1).upper()
        if val.isdigit():
            romans = {1: 'I', 2: 'II', 3: 'III', 4: 'IV', 5: 'V', 6: 'VI', 7: 'VII', 8: 'VIII', 9: 'IX', 10: 'X',
                      11: 'XI', 12: 'XII', 13: 'XIII', 14: 'XIV', 15: 'XV', 16: 'XVI', 17: 'XVII', 18: 'XVIII', 19: 'XIX', 20: 'XX'}
            val = romans.get(int(val), val)
        filter_conditions.append({"bab": {"$eq": f"BAB {val}"}})

    # 2. Deteksi BAGIAN (Gunakan $eq untuk pencocokan nilai persis)
    bagian_match = re.search(r'\bbagian\s+([a-z]+)\b', query_lower)
    if bagian_match:
        bagian_val = bagian_match.group(1).capitalize() # Contoh: "Kedua"
        filter_conditions.append({"bagian": {"$eq": f"Bagian {bagian_val}"}})

    # 3. Deteksi PASAL (Gunakan $eq untuk pencocokan nilai persis)
    pasal_match = re.search(r'\bpasal\s+(\d+)\b', query_lower)
    if pasal_match:
        pasal_num = pasal_match.group(1)
        filter_conditions.append({"pasal": {"$eq": f"Pasal {pasal_num}"}})

    # Rakit hasil filter akhir sesuai format ChromaDB
    if not filter_conditions:
        return {} 
    elif len(filter_conditions) == 1:
        return filter_conditions[0]
    else:
        # ChromaDB menggunakan format $and untuk menggabungkan banyak kondisi
        return {"$and": filter_conditions}

def search_duckduckgo_fallback(query: str, max_results: int = 10) -> str:
    """
    Fungsi cadangan pencarian web menggunakan DuckDuckGo.
    """
    try:
        with DDGS() as ddgs:
            raw_results = list(ddgs.text(query, max_results=max_results))
            cleaned_results = []
            
            for r in raw_results:
                body = r.get('body', '')
                if body:
                    cleaned_results.append(body)
                    
        if cleaned_results:
            return "SUMBER INFORMASI DARI INTERNET (DuckDuckGo):\n" + "\n\n".join(cleaned_results)
    except Exception as e:
        print(f"[ERROR] Gagal melakukan pencarian web: {e}")
        
    return "Informasi tidak ditemukan baik di dokumen lokal maupun di internet."

def invoke_hybrid_with_auto_filter(
    query,
    parent_retriever,
    compression_retriever,
    cross_encoder_model,
    threshold=0.0014
):
    """
    Menerapkan filter metadata secara otomatis berdasarkan kata kunci yang terdeteksi di query.
    """
    cleaned_query = normalize_to_roman(query)
    auto_filter = generate_dynamic_filter(cleaned_query)
    
    if auto_filter:        
        # Jika query spesifik (menyebut Bab/Pasal), langsung gunakan parent_retriever (vektor + filter) 
        parent_retriever.search_kwargs["filter"] = auto_filter
        
        # Bypass ensemble, langsung ambil dari compression_retriever yang basisnya parent_retriever
        docs = compression_retriever.invoke(cleaned_query)
    else:
        if "filter" in parent_retriever.search_kwargs:
                parent_retriever.search_kwargs.pop("filter")
        docs = compression_retriever.invoke(cleaned_query)

    # Hitung score
    if docs:
        text_pairs = [(cleaned_query, doc.page_content) for doc in docs]        
        scores = cross_encoder_model.score(text_pairs)
        
        top_score = float(max(scores))        
        if top_score < threshold:
            display_handle.update(
                Markdown(f"### 🤖 Jawaban AI:\n\n*Konteks tidak ditemukan, sistem melakukan pencarian umum di internet...*")
            )
            web_result_text = search_duckduckgo_fallback(cleaned_query)
            return [
                Document(
                    page_content=web_result_text,
                    metadata={"source": "DuckDuckGo Search"}
                )
            ]
        else:   
            top_docs = docs[0]
            return [
                Document(
                    page_content=top_docs.page_content,
                    metadata=top_docs.metadata
                )
            ]
    else:
        print("[WARN] Tidak ada dokumen lokal yang ditemukan.")
        web_result_text = search_duckduckgo_fallback(cleaned_query)
        return [
            Document(
                page_content=web_result_text,
                metadata={"source": "DuckDuckGo Search"}
            )
        ]

In [21]:
query = "apa isi dari bab 2 pasal 2 tahun 2023 ?"
cleaned_query = normalize_to_roman(query)
relevant_docs = invoke_hybrid_with_auto_filter(cleaned_query, parent_retriever, compression_retriever, cross_encoder_model)

for document in relevant_docs:
    print(document.metadata)
    print(document.page_content)
    print()

{'halaman': 14, 'dokumen': '/kaggle/working/Document RAG/UU Nomor 6 Tahun 2023.pdf', 'tingkat_hukum': 'UU ', 'nomor': '6', 'tahun': '2023', 'bab': 'BAB II', 'bagian': 'Bagian Umum', 'paragraf': '-', 'pasal': 'Pasal 2'}
[/kaggle/working/Document RAG/UU Nomor 6 Tahun 2023.pdf | BAB II | Bagian Umum | - | Pasal 2]
Pasal 2
(1) Peraturan Pemerintah Pengganti Undang-Undang ini
diselenggarakan berdasarkan asas:
a. pemerataan hak;
b. kepastian hukum;
c. kemudahan berusaha;
d. kebersamaan; dan
e. kemandirian.
(21 Selain berdasarkan asas sebagaimana dimaksud pada
ayat (1), penyelenggaraan Cipta Kerja dilaksanakan
berdasarkan asas lain sesuai dengan bidang hukum
yang diatur dalam undang-undang yang bersangkutan.
SK No 176015 A
Pasal3...



In [22]:
system_prompt = """
Anda adalah asisten hukum AI yang fokus pada pemecahan masalah (Problem Solver).

ATURAN WAJIB GENERASI:
1. Di dalam tag <think>...</think>: 
   - POLA PENALARAN: (1) Identifikasi poin pertanyaan user -> (2) Cocokkan dengan pasal/aturan yang relevan dari dokumen -> (3) Buat kesimpulan logis.
   - DILARANG MERANGKUM isi bab/pasal dari dokumen secara berurutan.
2. Di dalam tag <answer>...</answer>: 
   - Berikan jawaban langsung, jelas, dan solutif untuk user berdasarkan hasil analisis di <think>.
3. DILARANG MENULIS KATA DUPLIKAT Antara tag <think> dan <answer>.
"""

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def generate_with_llama3_template(inputs):
    context = inputs["context"]
    question = inputs["question"]
    
    messages = [
        {"role": "system", "content": system_prompt},
        
        # FEW-SHOT EXAMPLE (Pertanyaan di Atas, Konteks di Bawah)
        {
            "role": "user",
            "content": (
                "PERTANYAAN USER:\n"
                "Saya staf admin kerja lembur 3 jam, apakah berhak dapat uang lembur?\n\n"
                "DOKUMEN KONTEKS:\n"
                "PP No. 35 Tahun 2021 Pasal 27: Pengusaha wajib membayar upah lembur kepada pekerja/buruh yang dipekerjakan melebihi waktu kerja normal."
            )
        },
        {
            "role": "assistant",
            "content": (
                "<think>\n"
                "User menanyakan hak upah lembur untuk posisi staf admin (3 jam lembur). Berdasarkan PP No. 35 Tahun 2021 Pasal 27, kerja melebihi waktu normal wajib diberi upah lembur. Maka staf admin berhak atas upah 3 jam tersebut.\n"
                "</think>\n"
                "<answer>\n"
                "Ya, Anda berhak mendapatkan upah lembur. Berdasarkan PP No. 35 Tahun 2021 Pasal 27, pengusaha wajib membayar upah lembur untuk pekerja yang bekerja melebihi waktu kerja normal (8 jam sehari).\n"
                "</answer>"
            )
        },
        
        # REAL USER QUERY (Pertanyaan ditaruh PERTAMA agar LLM fokus memecahkan masalah)
        {
            "role": "user",
            "content": (
                f"PERTANYAAN USER:\n{question}\n\n"
                f"DOKUMEN KONTEKS:\n{context}"
            )
        }
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Assistant Prefill
    formatted_prompt += "<think>\n"
    return formatted_prompt


def clean_legal_response(raw_output: str) -> str:
    if not raw_output:
        return ""

    # Sisipkan tag <think> jika terpotong akibat prefill
    if not raw_output.startswith("<think>") and "</think>" in raw_output:
        raw_output = "<think>\n" + raw_output

    # Bersihkan token khusus Llama 3
    stop_tokens = ["<|start_header_id|>", "<|end_header_id|>", "<|eot_id|>", "<|end_of_text|>"]
    for token in stop_tokens:
        if token in raw_output:
            raw_output = raw_output.split(token)[0]

    # Ekstrak tag <think> dan <answer>
    think_match = re.search(r"<think>(.*?)</think>", raw_output, re.DOTALL)
    answer_match = re.search(r"<answer>(.*?)</answer>", raw_output, re.DOTALL)

    think_text = think_match.group(1).strip() if think_match else ""
    answer_text = answer_match.group(1).strip() if answer_match else ""

    # Jika tag <answer> tidak ada tapi ada teks setelah </think>, paksa jadi isi answer
    if not answer_text and "</think>" in raw_output:
        answer_text = raw_output.split("</think>")[-1].replace("<answer>", "").replace("</answer>", "").strip()

    if think_text and answer_text:
        return f"<think>\n{think_text}\n</think>\n<answer>\n{answer_text}\n</answer>"
    elif answer_text:
        return f"<answer>\n{answer_text}\n</answer>"
    
    return raw_output.strip()

In [23]:
rag_chain = (
    {
        "context": RunnableLambda(
            lambda query: invoke_hybrid_with_auto_filter(query, parent_retriever, compression_retriever, cross_encoder_model)
        ) | format_docs,
        
        "question": RunnablePassthrough()
    }
    | RunnableLambda(generate_with_llama3_template)
    | llm                                           
    | StrOutputParser()
    | RunnableLambda(clean_legal_response)
)

In [24]:
query = "apa isi bab 2 pasal 2 tahun 2023 ?"
cleaned_query = normalize_to_roman(query)
response = rag_chain.invoke(cleaned_query)
print(response)

<think>
User menanyakan isi pasal 2 dari UU No. 6 Tahun 2023. Berdasarkan dokumen, pasal 2 UU No. 6 Tahun 2023 berisi tentang asas penyelenggaraan Cipta Kerja. Pasal 2 ayat (1) menyebutkan asas pemerataan hak, kepastian hukum, kemudahan berusaha, kebersamaan, dan kemandirian. Pasal 2 ayat (2) menyebutkan bahwa penyelenggaraan Cipta Kerja dilaksanakan berdasarkan asas lain sesuai dengan bidang hukum yang diatur dalam undang-undang yang bersangkutan.
</think>
<answer>
Berdasarkan pasal 2 UU No. 6 Tahun 2023, penyelenggaraan Cipta Kerja dilaksanakan berdasarkan asas pemerataan hak, kepastian hukum, kemudahan berusaha, kebersamaan, dan kemandirian. Selain itu, penyelenggaraan Cipta Kerja dilaksanakan berdasarkan asas lain sesuai dengan bidang hukum yang diatur dalam undang-undang yang bersangkutan.
</answer>


In [25]:
def get_loading_ui_html():
    return """
    <div style="font-family: Arial, sans-serif; margin: 10px 0;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
            <div style="display: flex; align-items: center; gap: 8px;">
                <span style="font-size: 16px;">🤖 <b>Jawaban AI:</b></span>
                <span style="font-size: 13px; color: #555; font-style: italic;">Sedang menganalisis dokumen & menyusun penalaran...</span>
            </div>
            <span style="font-size: 12px; font-weight: bold; color: #2563eb; background: #eff6ff; padding: 3px 8px; border-radius: 4px; border: 1px solid #bfdbfe;" id="ai-timer-display">
                ⏱️ 00:00
            </span>
        </div>
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 6px; overflow: hidden; height: 8px; position: relative;">
            <div class="progress-bar-fill"></div>
        </div>
    </div>

    <style>
    @keyframes loading-slide {
        0% { left: -40%; width: 40%; }
        50% { left: 30%; width: 50%; }
        100% { left: 100%; width: 40%; }
    }

    .progress-bar-fill {
        position: absolute;
        height: 100%;
        background: linear-gradient(90deg, #2563eb, #3b82f6, #60a5fa);
        border-radius: 6px;
        animation: loading-slide 1.6s infinite ease-in-out;
    }
    </style>

    <script>
    (function() {
        var startTime = Date.now();
        var timerElem = document.getElementById("ai-timer-display");
        
        var timerInterval = setInterval(function() {
            // Hentikan timer jika elemen UI loading sudah diganti/dihapus dari DOM
            if (!document.getElementById("ai-timer-display")) {
                clearInterval(timerInterval);
                return;
            }
            var elapsedSeconds = Math.floor((Date.now() - startTime) / 1000);
            var minutes = String(Math.floor(elapsedSeconds / 60)).padStart(2, '0');
            var seconds = String(elapsedSeconds % 60).padStart(2, '0');
            if (timerElem) {
                timerElem.innerText = "⏱️ " + minutes + ":" + seconds;
            }
        }, 1000);
    })();
    </script>
    """

def remove_duplicate_paragraphs(text: str) -> str:
    """
    Menghapus paragraf atau kalimat yang diulang secara persis.
    """
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    seen = []
    
    for line in lines:
        # Hanya masukkan baris jika belum pernah muncul sebelumnya
        if line not in seen:
            seen.append(line)
            
    return "\n".join(seen)

def format_cot_for_markdown(response: str) -> str:
    think_match = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    answer_match = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)
    
    think_text = think_match.group(1).strip() if think_match else ""
    answer_text = answer_match.group(1).strip() if answer_match else response.strip()
    
    # Menghapus sisa tag jika ada
    answer_text = re.sub(r"</?answer>", "", answer_text).strip()

    
    # Hapus paragraf duplikat pada masing-masing bagian
    think_clean = remove_duplicate_paragraphs(think_text)
    answer_clean = remove_duplicate_paragraphs(answer_text)

    if think_text:
        # Ubah tiap baris penalaran menjadi kutipan (blockquote)
        formatted_think = "\n".join([f"> {line}" for line in think_clean.split("\n")])
        return f"#### 🧠 **Proses Penalaran:**\n{formatted_think}\n\n---\n#### 💬 **Jawaban Akhir:**\n{answer_clean}"

In [26]:
print("=" * 60)
print("=== Asisten Hukum AI   ===")
print("=== Ketik 'keluar' untuk berhenti ====")
print("=" * 60)

while True:
    try:
        query = input("\n[User Input] Masukkan Pertanyaan Anda: ").strip()
        cleaned_query = normalize_to_roman(query)

        if query.lower() in ["keluar"]:
            print("\n[INFO] Sesi diakhiri. Terima kasih!")
            break
            
        if not query:
            continue
            
        start_time = time.time()        
        display_handle = display(HTML(get_loading_ui_html()), display_id=True)
        response = rag_chain.invoke(cleaned_query)
        formatted_text_res = format_cot_for_markdown(response)        
        elapsed_sec = time.time() - start_time
        
        footer_info = f"\n\n---\n*⏱️ Selesai diproses dalam {elapsed_sec:.2f} detik*"
        display_handle.update(Markdown(f"### 🤖 Jawaban AI:\n\n{formatted_text_res}{footer_info}"))
        print("-" * 60)

    except KeyboardInterrupt:
        if display_handle is not None:
            display_handle.update(HTML(""))
        print("\n[INFO] Sesi dihentikan secara paksa oleh pengguna (Ctrl+C).")
        break
    except Exception as e:
        if display_handle is not None:
            display_handle.update(HTML(f"<div style='color: #dc2626; margin: 10px 0;'>❌ <b>Terjadi kesalahan:</b> {e}</div>"))
        else:
            print(f"\n[ERROR] Terjadi kesalahan saat memproses: {e}")

=== Asisten Hukum AI   ===
=== Ketik 'keluar' untuk berhenti ====



[User Input] Masukkan Pertanyaan Anda:  apa isi dari bab 2 pasal 2 ?


### 🤖 Jawaban AI:

#### 🧠 **Proses Penalaran:**
> User menanyakan isi dari bab II pasal 2. Berdasarkan dokumen, bab II pasal 2 berisi tentang asas penyelenggaraan Cipta Kerja. Pasal 2 ayat (1) menyebutkan bahwa peraturan pemerintah pengganti undang-undang ini diselenggarakan berdasarkan asas pemerataan hak, kepastian hukum, kemudahan berusaha, kebersamaan, dan kemandirian. Pasal 2 ayat (2) menyebutkan bahwa selain berdasarkan asas sebagaimana dimaksud pada ayat (1), penyelenggaraan Cipta Kerja dilaksanakan berdasarkan asas lain sesuai dengan bidang hukum yang diatur dalam undang-undang yang bersangkutan.

---
#### 💬 **Jawaban Akhir:**
Berdasarkan dokumen, bab II pasal 2 berisi tentang asas penyelenggaraan Cipta Kerja. Pasal 2 ayat (1) menyebutkan bahwa peraturan pemerintah pengganti undang-undang ini diselenggarakan berdasarkan asas pemerataan hak, kepastian hukum, kemudahan berusaha, kebersamaan, dan kemandirian. Pasal 2 ayat (2) menyebutkan bahwa selain berdasarkan asas sebagaimana dimaksud pada ayat (1), penyelenggaraan Cipta Kerja dilaksanakan berdasarkan asas lain sesuai dengan bidang hukum yang diatur dalam undang-undang yang bersangkutan.

---
*⏱️ Selesai diproses dalam 146.98 detik*

------------------------------------------------------------



[User Input] Masukkan Pertanyaan Anda:  siapa hendri mardani ?


### 🤖 Jawaban AI:

#### 🧠 **Proses Penalaran:**
> Dari sumber informasi di atas, dapat disimpulkan bahwa siapa hendri mardani adalah seorang engineer, data scientist, dan data analyst. Ia juga merupakan anggota dari Dicoding Elite dan External Code Reviewer. Dari informasi tersebut, dapat disimpulkan bahwa siapa hendri mardani adalah seorang yang berdedikasi dan berkompeten dalam bidang teknologi informasi.

---
#### 💬 **Jawaban Akhir:**
Dari sumber informasi di atas, dapat disimpulkan bahwa siapa hendri mardani adalah seorang engineer, data scientist, dan data analyst. Ia juga merupakan anggota dari Dicoding Elite dan External Code Reviewer. Dari informasi tersebut, dapat disimpulkan bahwa siapa hendri mardani adalah seorang yang berdedikasi dan berkompeten dalam bidang teknologi informasi.

---
*⏱️ Selesai diproses dalam 121.36 detik*

------------------------------------------------------------



[User Input] Masukkan Pertanyaan Anda:  keluar



[INFO] Sesi diakhiri. Terima kasih!
